# recs_021 — V2a logpop_blend head-to-head

**Val-only face-off:** frozen `two_tower_v1` @100 pools → rerank @10.

Compare the two promotion candidates that beat D1 on `val_dev_12k_v1`:

| Method | Source | Metadata signal | Fixed train_tune config |
|--------|--------|-----------------|-------------------------|
| `two_tower_v1_v2a_query_metadata_logpop_blend` | [`recs_019`](recs_019_v2a_metadata_jaccard.ipynb) | Jaccard FK sets | `w_meta=0.05`, `genre_theme_kw`, query anchor |
| `two_tower_v1_v2a_embed_query_logpop_blend` | [`recs_020`](recs_020_v2a_taxonomy_use_cosine.ipynb) | taxonomy USE pooled cosine | `w_meta=0.1`, `genre_theme_kw`, pooled, query anchor |

**Reference baselines:** D1 (`two_tower_v1_heuristic_logpop_blend`), bare pool, oracle.

**No re-tuning** — hyperparameters are frozen from parent spikes. Goal: pick ship candidate for eval-job wiring.

**Prereqs:** Parent notebooks run; `artifacts/igdb/igdb_games__enriched.parquet` present.

# Executive Summary

**Question:**  
Between Jaccard and taxonomy-USE logpop_blend winners, which should replace D1 on val?

**Result:**  
On `val_dev_12k_v1` (12.5k examples, frozen `two_tower_v1` @100 pools):

| Method | NDCG@10 overall | Slice A | Pers gap vs pop |
|--------|-----------------|---------|-----------------|
| **`two_tower_v1_v2a_embed_query_logpop_blend`** | **0.095** | **0.070** | **0.726** |
| `two_tower_v1_v2a_query_metadata_logpop_blend` | 0.095 | 0.070 | 0.725 |
| D1 `two_tower_v1_heuristic_logpop_blend` | 0.093 | 0.068 | 0.720 |

Pairwise NDCG@10 (embed vs Jaccard): **617 embed wins / 533 Jaccard wins** (11,350 ties); Slice A **56 / 48** embed wins; mean Δ **+0.00017** overall.

**Recommendation / Decision:**  
**Ship `two_tower_v1_v2a_embed_query_logpop_blend`** as v2a ranker (`w_meta=0.1`, pooled USE, `genre_theme_kw`, query anchor). Kill Jaccard logpop_blend for production. Wire into eval jobs + `pool_rerank_registry`.

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable, Literal

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import (
    DEFAULT_LOGPOP_BLEND_ALPHA,
    minmax_norm,
    score_logpop_blend,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    RANKING_REPORT_METRIC_COLS,
    _append_personalization_metrics,
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_by_slice_for_metrics,
    _table_overall_ranking,
    _table_personalization,
    average_precision_at_k,
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)
from steam_review_ml.igdb.constants import TAXONOMY_RESOLVE_FIELDS
from steam_review_ml.recommender.retrieve import ContentRetriever

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "two_tower_v1"
K_FINAL = 10
K_PERSONALIZATION = 10
MIN_REVIEW_CHARS = 30
D1_ALPHA = DEFAULT_LOGPOP_BLEND_ALPHA

VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_EXAMPLES_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
IGDB_ENRICHED = REPO_ROOT / "artifacts/igdb/igdb_games__enriched.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPIKE_OUT = ARTIFACT_DIR / "spikes/v2a_head_to_head"

TAXONOMY_FIELDS: tuple[str, ...] = TAXONOMY_RESOLVE_FIELDS
POOLED_COL = {f: f"{f}_names__use_pooled" for f in TAXONOMY_FIELDS}
ENTITY_COL = {f: f"{f}_names__use" for f in TAXONOMY_FIELDS}

# --- Frozen winner configs (from parent spikes; do not tune here) ---
METHOD_JACCARD_LOGPOP = "two_tower_v1_v2a_query_metadata_logpop_blend"
METHOD_EMBED_LOGPOP = "two_tower_v1_v2a_embed_query_logpop_blend"
METHOD_D1 = "two_tower_v1_heuristic_logpop_blend"

WINNER_FIELDS: tuple[str, ...] = ("genres", "themes", "keywords")
JACCARD_W_META = 0.05
EMBED_W_META = 0.1
EMBED_SIM_MODE: Literal["pooled"] = "pooled"

for p in (VAL_JSONL, VAL_EXAMPLES_PARQUET, IGDB_ENRICHED):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

SPIKE_OUT.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SPIKE_OUT={SPIKE_OUT}")

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-22 17:22:25.779949: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-22 17:22:25.792425: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782163345.806720 3585971 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782163345.811420 3585971 cuda_bl

REPO_ROOT=/home/ryanr/workspace/steam_recommendations
SPIKE_OUT=/home/ryanr/workspace/steam_recommendations/artifacts/recs/spikes/v2a_head_to_head


## Load data

In [2]:
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
val_pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
X_emb = retriever.embedding_matrix

val_examples_df = pd.read_parquet(VAL_EXAMPLES_PARQUET)
examples_for_pers: list[dict[str, Any]] = []
for _, row in val_examples_df.iterrows():
    examples_for_pers.append(
        {
            "ex_idx": int(row["ex_idx"]),
            "user_id": row["user_id"],
            "query_app_id": int(row["query_app_id"]),
            "query_ts": float(row["query_ts"]),
            "n_eval_targets": int(row["n_eval_targets"]),
            "train_review_rows": json.loads(row["train_review_rows_json"]),
            "validation_positive_app_ids": json.loads(row["validation_positive_app_ids_json"]),
        }
    )

print(f"val pools: {len(val_pools):,}  catalog apps: {len(app_ids):,}")

val pools: 12,500  catalog apps: 315


In [3]:
def parse_fk_set(val: Any) -> frozenset[int]:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return frozenset()
    if isinstance(val, (list, tuple, np.ndarray)):
        return frozenset(int(x) for x in val)
    return frozenset()


def parse_pooled_use(val: Any) -> np.ndarray:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.zeros(512, dtype=np.float64)
    arr = np.asarray(val, dtype=np.float64)
    return arr.reshape(512) if arr.size == 512 else np.zeros(512, dtype=np.float64)


igdb_fk = pd.read_parquet(IGDB_ENRICHED, columns=["app_id", *TAXONOMY_FIELDS])
field_sets_by_app: dict[int, dict[str, frozenset[int]]] = {}
for _, row in igdb_fk.iterrows():
    app_id = int(row["app_id"])
    field_sets_by_app[app_id] = {f: parse_fk_set(row[f]) for f in TAXONOMY_FIELDS}

use_cols = [POOLED_COL[f] for f in TAXONOMY_FIELDS]
igdb_use = pd.read_parquet(IGDB_ENRICHED, columns=["app_id", *use_cols])
pooled_by_app: dict[int, dict[str, np.ndarray]] = {}
for _, row in igdb_use.iterrows():
    app_id = int(row["app_id"])
    pooled_by_app[app_id] = {f: parse_pooled_use(row[POOLED_COL[f]]) for f in TAXONOMY_FIELDS}

print(f"IGDB apps: fk={len(field_sets_by_app):,}  use_pooled={len(pooled_by_app):,}")

IGDB apps: fk=315  use_pooled=315


## Scoring (frozen winner formulas)

In [4]:
def jaccard(a: frozenset[int], b: frozenset[int]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na <= 1e-12 or nb <= 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def anchor_fk_sets(query_app_id: int) -> dict[str, frozenset[int]]:
    per_app = field_sets_by_app.get(int(query_app_id), {})
    return {f: per_app.get(f, frozenset()) for f in TAXONOMY_FIELDS}


def jaccard_meta_scores(pool_app_ids: list[int], *, query_app_id: int) -> np.ndarray:
    anchor = anchor_fk_sets(query_app_id)
    meta = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        cand = field_sets_by_app.get(int(app_id), {})
        meta[i] = float(np.mean([jaccard(anchor[f], cand.get(f, frozenset())) for f in WINNER_FIELDS]))
    return meta


def embed_meta_scores(pool_app_ids: list[int], *, query_app_id: int) -> np.ndarray:
    meta = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        sims = []
        for f in WINNER_FIELDS:
            anchor = pooled_by_app.get(int(query_app_id), {}).get(f, np.zeros(512))
            cand = pooled_by_app.get(int(app_id), {}).get(f, np.zeros(512))
            sims.append(cosine_sim(anchor, cand))
        meta[i] = float(np.mean(sims))
    return meta


def logpop_blend_meta(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    w_meta: float,
    meta_scores: np.ndarray,
) -> np.ndarray:
    """``(1 - w_meta) * norm(D1) + w_meta * norm(meta)``; w_meta=0 → pure D1."""
    d1 = score_logpop_blend(
        pool_app_ids,
        retrieval_scores,
        alpha=D1_ALPHA,
        pop_row=pop_row,
        app_to_row=app_to_row,
    )
    if w_meta <= 0.0:
        return d1
    if w_meta >= 1.0:
        return minmax_norm(meta_scores)
    return (1.0 - w_meta) * minmax_norm(d1) + w_meta * minmax_norm(meta_scores)


def pool_scores_to_ranked_indices(pool_app_ids: list[int], pool_scores: np.ndarray) -> np.ndarray:
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:K_FINAL]

## Val face-off

In [5]:
def score_pool(row: dict[str, Any], method: str) -> np.ndarray:
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = json.loads(row["retrieved_scores_json"])
    qid = int(row["query_app_id"])
    if method == METHOD_D1:
        return score_logpop_blend(pool_apps, ret_sc, alpha=D1_ALPHA, pop_row=pop_row, app_to_row=app_to_row)
    if method == METHOD_JACCARD_LOGPOP:
        meta = jaccard_meta_scores(pool_apps, query_app_id=qid)
        return logpop_blend_meta(pool_apps, ret_sc, w_meta=JACCARD_W_META, meta_scores=meta)
    if method == METHOD_EMBED_LOGPOP:
        meta = embed_meta_scores(pool_apps, query_app_id=qid)
        return logpop_blend_meta(pool_apps, ret_sc, w_meta=EMBED_W_META, meta_scores=meta)
    raise ValueError(method)


def per_example_row(row: dict[str, Any], *, method: str) -> dict[str, Any] | None:
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)
    oracle_indices = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)
    if method == f"{POOL_METHOD}_oracle":
        ranked = oracle_indices[:K_FINAL]
    elif method == POOL_METHOD:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(json.loads(row["retrieved_scores_json"])))
    else:
        blend = score_pool(row, method)
        ranked = pool_scores_to_ranked_indices(pool_apps, blend)
    return {
        "method": method,
        "ex_idx": int(row["ex_idx"]),
        "slice_name": row["slice_name"],
        "n_eval_targets": int(row["n_eval_targets"]),
        "query_app_id": int(row["query_app_id"]),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "Precision@K": precision_at_k(ranked, positives, K_FINAL, app_ids),
        "Recall@K": recall_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
        "OracleHit@K": hit_rate_at_k(oracle_indices, positives, K_FINAL, app_ids),
        "OracleNDCG@K": ndcg_at_k(oracle_indices, positives, K_FINAL, app_ids),
        "ranked_app_ids": [int(app_ids[i]) for i in ranked],
    }


METHODS = (
    METHOD_JACCARD_LOGPOP,
    METHOD_EMBED_LOGPOP,
    METHOD_D1,
    POOL_METHOD,
    f"{POOL_METHOD}_oracle",
)

rows: list[dict[str, Any]] = []
for row in val_pools:
    for method in METHODS:
        mrow = per_example_row(row, method=method)
        if mrow:
            rows.append(mrow)

df_val = pd.DataFrame(rows)
df_metrics = df_val.drop(columns=["ranked_app_ids"])
df_metrics.to_parquet(SPIKE_OUT / "v2a_head_to_head_val_per_example.parquet", index=False)

overall = _table_overall_ranking(df_metrics)
by_slice = _table_by_slice_for_metrics(df_metrics, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)
overall.to_csv(SPIKE_OUT / "v2a_head_to_head_val_overall.csv", index=False)
by_slice.to_csv(SPIKE_OUT / "v2a_head_to_head_val_by_slice.csv", index=False)

display(Markdown("### Val overall"))
display(overall.sort_values("NDCG@K", ascending=False))
display(Markdown("### Val by slice (winners + D1)"))
winner_methods = {METHOD_JACCARD_LOGPOP, METHOD_EMBED_LOGPOP, METHOD_D1}
display(by_slice[by_slice["method"].isin(winner_methods)].sort_values(["slice_name", "NDCG@K"], ascending=[True, False]))

### Val overall

,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
0,two_tower_v1_oracle,0.51224,0.054144,0.494053,0.494053,0.498310,0.512240,0.51224,0.49831
1,two_tower_v1_v2a_embed_query_logpop_blend,0.19584,0.019816,0.186324,0.066797,0.095345,0.069567,0.51224,0.49831
2,two_tower_v1_v2a_query_metadata_logpop_blend,0.19664,0.019912,0.187432,0.066291,0.095176,0.069114,0.51224,0.49831
3,two_tower_v1_heuristic_logpop_blend,0.19328,0.019560,0.184095,0.064321,0.092892,0.067059,0.51224,0.49831
4,two_tower_v1,0.04680,0.004728,0.043740,0.010325,0.018161,0.011008,0.51224,0.49831


### Val by slice (winners + D1)

,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
1,slice_a_multi_target,two_tower_v1_v2a_embed_query_logpop_blend,0.281379,0.032138,0.117319,0.035030,0.070334,0.082787,0.773793,0.533618
2,slice_a_multi_target,two_tower_v1_v2a_query_metadata_logpop_blend,0.273103,0.031586,0.114344,0.035371,0.069861,0.084047,0.773793,0.533618
3,slice_a_multi_target,two_tower_v1_heuristic_logpop_blend,0.271724,0.031172,0.113367,0.034181,0.068322,0.081396,0.773793,0.533618
6,slice_b_single_target,two_tower_v1_v2a_embed_query_logpop_blend,0.190573,0.019057,0.190573,0.068753,0.096885,0.068753,0.496136,0.496136
7,slice_b_single_target,two_tower_v1_v2a_query_metadata_logpop_blend,0.191932,0.019193,0.191932,0.068195,0.096735,0.068195,0.496136,0.496136
8,slice_b_single_target,two_tower_v1_heuristic_logpop_blend,0.188450,0.018845,0.188450,0.066176,0.094404,0.066176,0.496136,0.496136


## Pairwise: Jaccard vs embed logpop_blend

In [6]:
wide = df_val.pivot_table(index=["ex_idx", "slice_name", "n_eval_targets"], columns="method", values="NDCG@K")
wide = wide.dropna(subset=[METHOD_JACCARD_LOGPOP, METHOD_EMBED_LOGPOP])
wide["delta_embed_minus_jaccard"] = wide[METHOD_EMBED_LOGPOP] - wide[METHOD_JACCARD_LOGPOP]
wide["embed_wins"] = wide["delta_embed_minus_jaccard"] > 0
wide["jaccard_wins"] = wide["delta_embed_minus_jaccard"] < 0
wide["tie"] = wide["delta_embed_minus_jaccard"] == 0

pair_summary = pd.Series(
    {
        "n_examples": len(wide),
        "embed_wins": int(wide["embed_wins"].sum()),
        "jaccard_wins": int(wide["jaccard_wins"].sum()),
        "ties": int(wide["tie"].sum()),
        "mean_delta_embed_minus_jaccard": float(wide["delta_embed_minus_jaccard"].mean()),
    },
    name="pairwise_ndcg",
)
display(pair_summary)

slice_a = wide[wide.index.get_level_values("slice_name") == "slice_a_multi_target"]
if len(slice_a):
    display(Markdown("### Pairwise on Slice A only"))
    display(
        pd.Series(
            {
                "embed_wins": int(slice_a["embed_wins"].sum()),
                "jaccard_wins": int(slice_a["jaccard_wins"].sum()),
                "mean_delta": float(slice_a["delta_embed_minus_jaccard"].mean()),
            }
        )
    )

wide.reset_index().to_csv(SPIKE_OUT / "v2a_head_to_head_pairwise_ndcg.csv", index=False)

n_examples                        12500.000000
embed_wins                          617.000000
jaccard_wins                        533.000000
ties                              11350.000000
mean_delta_embed_minus_jaccard        0.000169
Name: pairwise_ndcg, dtype: float64

### Pairwise on Slice A only

embed_wins      56.000000
jaccard_wins    48.000000
mean_delta       0.000473
dtype: float64

## Personalization

In [7]:
def popularity_train_score(ex: dict[str, Any]) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(ex["query_app_id"]))
    if row is not None:
        s[row] = -np.inf
    return s.astype(np.float32)


def full_catalog_score(ex: dict[str, Any], method: str) -> np.ndarray:
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    blend = score_pool(row, method)
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, sc in zip(pool_apps, blend):
        full[int(app_to_row[int(app_id)])] = float(sc)
    return full.astype(np.float32)


pers_methods = {
    "popularity_train": popularity_train_score,
    METHOD_JACCARD_LOGPOP: lambda ex: full_catalog_score(ex, METHOD_JACCARD_LOGPOP),
    METHOD_EMBED_LOGPOP: lambda ex: full_catalog_score(ex, METHOD_EMBED_LOGPOP),
    METHOD_D1: lambda ex: full_catalog_score(ex, METHOD_D1),
}

personalization = _table_personalization(
    methods=pers_methods,
    examples=examples_for_pers,
    X=X_emb,
    app_ids=app_ids,
    pop_row=pop_row,
    k_personalization=K_PERSONALIZATION,
)
overall_pers = _append_personalization_metrics(overall.copy(), personalization, on_keys=["method"])
personalization.to_csv(SPIKE_OUT / "v2a_head_to_head_personalization.csv", index=False)
overall_pers.to_csv(SPIKE_OUT / "v2a_head_to_head_overall_with_personalization.csv", index=False)

display(personalization.sort_values("PersonalizationGapVsPopularity@10", ascending=False))

,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
2,two_tower_v1_v2a_embed_query_logpop_blend,0.202787,0.511111,6.296105,0.726256
3,two_tower_v1_v2a_query_metadata_logpop_blend,0.203304,0.517460,6.291719,0.724877
1,two_tower_v1_heuristic_logpop_blend,0.204632,0.501587,6.272161,0.720097
0,popularity_train,0.212335,0.034921,5.317471,0.000000


# Key Findings

**Finding 1:** Both logpop_blend winners beat D1 on overall NDCG@10 (**0.095 vs 0.093**). Embed edges Jaccard on overall (**0.095345 vs 0.095176**).

**Finding 2:** Slice A (promotion co-gate) — embed **0.070334** vs Jaccard **0.069861** vs D1 **0.068322**.

**Finding 3:** Pairwise per-example wins favor embed (**617 vs 533** overall; **56 vs 48** on Slice A). Most examples tie (identical NDCG); embed wins the marginal cases.

**Finding 4:** Personalization gap vs popularity meets guardrail for both winners (**0.726** embed, **0.725** Jaccard) vs D1 **0.720**.

# Recommendation / Next Steps

**Recommended Action:**  
Ship **`two_tower_v1_v2a_embed_query_logpop_blend`** — wired in `pool_rerank_registry`, `recs_job_eval_ranking.json`, and `recs_job_eval_offline.json`. Kill Jaccard logpop_blend for production.

**Tie-break applied:** Metrics within noise overall; embed wins Slice A + pairwise count → prefer taxonomy USE (richer signal than FK Jaccard).

**Artifacts:** `artifacts/recs/spikes/v2a_head_to_head/`